In [5]:
import math
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

try:
    from scipy.stats import chi2
except Exception:
    chi2 = None

# Paths
BASE_DIR = Path(".")
INPUT_CSV = BASE_DIR / "IPIP-FFM-data-8Nov2018" / "data-final.csv"
FLAGGED_OUTPUT_CSV = BASE_DIR / "output" / "data-flagged-qc.csv"
CLEANED_OUTPUT_CSV = BASE_DIR / "output" / "human_data_cleaned.csv"
QC_SUMMARY_CSV = BASE_DIR / "output" / "qc-flag-summary.csv"
QC_OVERLAP_CSV = BASE_DIR / "output" / "qc-flag-overlap.csv"

# Thresholds / policy
SPEEDING_MEAN_SECONDS_THRESHOLD = 1.5
STRAIGHTLINING_MAX_UNIQUE = 2
LONGSTRING_THRESHOLD = 10
MAHALANOBIS_ALPHA = 0.001  # p < .001
IQR_MULTIPLIER = 1.5

# Column groups
TRAITS = ["EXT", "EST", "AGR", "CSN", "OPN"]
ITEM_COLS = [f"{trait}{i}" for trait in TRAITS for i in range(1, 11)]
TIMING_COLS = [f"{trait}{i}_E" for trait in TRAITS for i in range(1, 11)]

# Presented order in codebook: EXT1, AGR1, CSN1, EST1, OPN1, EXT2, ...
PRESENTATION_TRAIT_ORDER = ["EXT", "AGR", "CSN", "EST", "OPN"]
ITEM_COLS_PRESENTED = [
    f"{trait}{i}"
    for i in range(1, 11)
    for trait in PRESENTATION_TRAIT_ORDER
]

# Context variables used for plausibility and outlier checks.
CONTEXT_COLS = ["screenw", "screenh", "testelapse"]

print("QC pipeline configured")
print(f"Input exists: {INPUT_CSV.exists()} -> {INPUT_CSV}")
print(f"Item cols: {len(ITEM_COLS)} | Timing cols: {len(TIMING_COLS)} | Context cols: {len(CONTEXT_COLS)}")
print(f"Thresholds: speeding<{SPEEDING_MEAN_SECONDS_THRESHOLD}s, straightlining<={STRAIGHTLINING_MAX_UNIQUE} unique, longstring>={LONGSTRING_THRESHOLD}, Mahalanobis alpha={MAHALANOBIS_ALPHA}")


QC pipeline configured
Input exists: True -> IPIP-FFM-data-8Nov2018/data-final.csv
Item cols: 50 | Timing cols: 50 | Context cols: 3
Thresholds: speeding<1.5s, straightlining<=2 unique, longstring>=10, Mahalanobis alpha=0.001


In [6]:
def max_longstring(values: np.ndarray) -> int:
    """Return max run length of identical consecutive responses."""
    if values.size == 0:
        return 0

    max_run = 1
    current_run = 1
    last = values[0]

    for v in values[1:]:
        if pd.isna(v) or pd.isna(last):
            current_run = 1
        elif v == last:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1
        last = v

    return int(max_run)


def mahalanobis_d2(z: np.ndarray) -> np.ndarray:
    """Compute Mahalanobis distance squared with stable pseudo-inverse covariance."""
    cov = np.cov(z, rowvar=False)
    ridge = 1e-6
    cov_reg = cov + np.eye(cov.shape[0]) * ridge
    cov_inv = np.linalg.pinv(cov_reg)
    d2 = np.einsum("ij,jk,ik->i", z, cov_inv, z)
    return d2


def chi2_cutoff(df: int, alpha: float) -> float:
    """Return chi-square upper-tail cutoff at 1-alpha."""
    if chi2 is not None:
        return float(chi2.ppf(1 - alpha, df))

    # Fallback using Wilson-Hilferty approximation if scipy is unavailable.
    z = 3.09023230616781  # ~N(0,1) quantile for 0.999
    return float(df * (1 - 2 / (9 * df) + z * math.sqrt(2 / (9 * df))) ** 3)


In [7]:
# Load the raw survey export and create a working copy for QC processing.
print("Step 1/2: loading data and computing response-level QC metrics...")
step1_start = perf_counter()
raw = pd.read_csv(INPUT_CSV, sep="\t")
df = raw.copy()
print(f"Loaded raw data: {len(df):,} rows x {len(df.columns):,} columns")

# Ensure the columns required by the QC pipeline are present before proceeding.
required_cols = set(ITEM_COLS + TIMING_COLS + ["IPC"] + CONTEXT_COLS)
missing_required = sorted(required_cols - set(df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Convert only QC-relevant columns to numeric values in one batch.
numeric_cols = ITEM_COLS + TIMING_COLS + ["IPC"] + CONTEXT_COLS
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
numeric_missing_total = int(df[numeric_cols].isna().sum().sum())
print(f"Numeric coercion complete for {len(numeric_cols)} columns; total missing numeric values: {numeric_missing_total:,}")

# Use questionnaire presentation order for sequence-based checks.
presented_values = df[ITEM_COLS_PRESENTED].to_numpy(dtype=float)

# Build derived QC columns separately, then attach once to avoid fragmentation.
qc = pd.DataFrame(index=df.index)

# Technical flags
qc["flag_ipc_duplicate"] = df["IPC"].ne(1)
qc["flag_missing_items"] = df[ITEM_COLS].isna().any(axis=1)
qc["flag_item_out_of_range"] = ((df[ITEM_COLS] < 1) | (df[ITEM_COLS] > 5)).any(axis=1)
print(f"Technical flags: IPC duplicates={int(qc['flag_ipc_duplicate'].sum()):,}, missing items={int(qc['flag_missing_items'].sum()):,}, out-of-range items={int(qc['flag_item_out_of_range'].sum()):,}")

# Engagement and pattern flags
qc["mean_item_time_seconds"] = df[TIMING_COLS].mean(axis=1) / 1000.0
qc["flag_speeding"] = qc["mean_item_time_seconds"] < SPEEDING_MEAN_SECONDS_THRESHOLD

qc["item_unique_count"] = df[ITEM_COLS].nunique(axis=1, dropna=True)
qc["item_variance"] = df[ITEM_COLS].var(axis=1, ddof=1)
qc["flag_straightlining"] = qc["item_unique_count"] <= STRAIGHTLINING_MAX_UNIQUE

qc["max_longstring"] = [max_longstring(row) for row in presented_values]
qc["flag_longstring"] = qc["max_longstring"] >= LONGSTRING_THRESHOLD

print(f"Pattern flags: speeding={int(qc['flag_speeding'].sum()):,}, straightlining={int(qc['flag_straightlining'].sum()):,}, longstring={int(qc['flag_longstring'].sum()):,}")

# Mahalanobis D^2 is only computed on complete, in-range item responses.
item_valid = (~qc["flag_missing_items"]) & (~qc["flag_item_out_of_range"])
mahalanobis_d2_series = pd.Series(np.nan, index=df.index, dtype=float)
if item_valid.any():
    complete = df.loc[item_valid, ITEM_COLS].to_numpy(dtype=float)
    col_means = complete.mean(axis=0)
    col_stds = complete.std(axis=0, ddof=1)
    col_stds[col_stds == 0] = 1.0
    complete_z = (complete - col_means) / col_stds
    mahalanobis_d2_series.loc[item_valid] = mahalanobis_d2(complete_z)

qc["mahalanobis_d2"] = mahalanobis_d2_series
cutoff = chi2_cutoff(df=len(ITEM_COLS), alpha=MAHALANOBIS_ALPHA)
qc["mahalanobis_cutoff"] = cutoff
qc["flag_mahalanobis"] = qc["mahalanobis_d2"].gt(cutoff)
print(f"Mahalanobis scored on {int(item_valid.sum()):,} complete rows; flagged={int(qc['flag_mahalanobis'].sum()):,} at cutoff {cutoff:.3f}")

df = pd.concat([df, qc], axis=1)

print(f"Step 1 complete in {perf_counter() - step1_start:.2f}s")
print(f"Working DataFrame now has {len(df.columns):,} columns")


Step 1/2: loading data and computing response-level QC metrics...
Loaded raw data: 1,015,341 rows x 110 columns
Numeric coercion complete for 104 columns; total missing numeric values: 184,215
Technical flags: IPC duplicates=318,496, missing items=1,783, out-of-range items=139,124
Pattern flags: speeding=11,215, straightlining=8,787, longstring=10,766
Mahalanobis scored on 874,434 complete rows; flagged=61,208 at cutoff 86.661
Step 1 complete in 21.14s
Working DataFrame now has 123 columns


In [8]:
print("Step 2/2: applying context checks, exclusions, and writing outputs...")
step2_start = perf_counter()

# Build context-based outlier and plausibility columns separately for clarity.
context_qc = pd.DataFrame(index=df.index)

for col in CONTEXT_COLS:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - IQR_MULTIPLIER * iqr
    high = q3 + IQR_MULTIPLIER * iqr

    context_qc[f"{col}_iqr_low"] = low
    context_qc[f"{col}_iqr_high"] = high
    context_qc[f"flag_{col}_outlier"] = df[col].lt(low) | df[col].gt(high)
    context_qc[f"flag_{col}_nonpositive"] = df[col].le(0) | df[col].isna()

    print(
        f"  {col}: IQR bounds [{low:.2f}, {high:.2f}] | "
        f"outliers={int(context_qc[f'flag_{col}_outlier'].sum()):,} | "
        f"nonpositive/missing={int(context_qc[f'flag_{col}_nonpositive'].sum()):,}"
    )

df = pd.concat([df, context_qc], axis=1)

# A row is excluded if it trips any of the QC flags below.
exclude_flags = [
    "flag_ipc_duplicate",
    "flag_missing_items",
    "flag_item_out_of_range",
    "flag_speeding",
    "flag_straightlining",
    "flag_longstring",
    "flag_mahalanobis",
    "flag_screenw_outlier",
    "flag_screenh_outlier",
    "flag_testelapse_outlier",
]

df["exclude_any"] = df[exclude_flags].any(axis=1)
print(f"Exclusion summary: excluded={int(df['exclude_any'].sum()):,} / {len(df):,} ({df['exclude_any'].mean():.2%})")

# Save both the fully flagged file and the cleaned analytic subset.
cleaned = df.loc[~df["exclude_any"]].copy()

flag_cols = exclude_flags + ["exclude_any"]

summary = pd.DataFrame(
    {
        "flag": flag_cols,
        "count": [int(df[c].sum()) for c in flag_cols],
    }
)
summary["pct"] = summary["count"] / len(df)

overlap = df[exclude_flags].astype(int).corr()

df.to_csv(FLAGGED_OUTPUT_CSV, index=False)
cleaned.to_csv(CLEANED_OUTPUT_CSV, index=False)
summary.to_csv(QC_SUMMARY_CSV, index=False)
overlap.to_csv(QC_OVERLAP_CSV, index=True)

print(f"Flagged dataset written: {FLAGGED_OUTPUT_CSV}")
print(f"Cleaned dataset written: {CLEANED_OUTPUT_CSV}")
print(f"QC summary written: {QC_SUMMARY_CSV}")
print(f"QC overlap matrix written: {QC_OVERLAP_CSV}")
print(f"Rows kept: {len(cleaned):,} / {len(df):,} ({len(cleaned) / len(df):.2%})")
print(f"Step 2 complete in {perf_counter() - step2_start:.2f}s")

summary.sort_values("count", ascending=False).head(15)


Step 2/2: applying context checks, exclusions, and writing outputs...
  screenw: IQR bounds [-1125.00, 2979.00] | outliers=1,001 | nonpositive/missing=4,790
  screenh: IQR bounds [450.00, 1170.00] | outliers=30,836 | nonpositive/missing=4,790
  testelapse: IQR bounds [-42.00, 526.00] | outliers=88,901 | nonpositive/missing=1,783
Exclusion summary: excluded=514,638 / 1,015,341 (50.69%)


/var/folders/wt/vk4hp_7s3p59n11ck8n3b0640000gn/T/ipykernel_30010/1316622151.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["exclude_any"] = df[exclude_flags].any(axis=1)


Flagged dataset written: output/data-flagged-qc.csv
Cleaned dataset written: output/human_data_cleaned.csv
QC summary written: output/qc-flag-summary.csv
QC overlap matrix written: output/qc-flag-overlap.csv
Rows kept: 500,703 / 1,015,341 (49.31%)
Step 2 complete in 41.03s


,flag,count,pct
10,exclude_any,514638,0.506862
0,flag_ipc_duplicate,318496,0.313684
2,flag_item_out_of_range,139124,0.137022
9,flag_testelapse_outlier,88901,0.087558
6,flag_mahalanobis,61208,0.060283
8,flag_screenh_outlier,30836,0.030370
3,flag_speeding,11215,0.011046
5,flag_longstring,10766,0.010603
4,flag_straightlining,8787,0.008654
1,flag_missing_items,1783,0.001756


In [9]:
# Inspect the distribution of the main continuous QC metrics after scoring.
print("Advanced metric diagnostics:")
diagnostics = cleaned[[
    "mean_item_time_seconds",
    "item_unique_count",
    "max_longstring",
    "mahalanobis_d2",
]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
diagnostics


Advanced metric diagnostics:


,mean_item_time_seconds,item_unique_count,max_longstring,mahalanobis_d2
count,500703.000000,500703.000000,500703.000000,500703.000000
mean,4.562165,4.904812,3.549439,45.213231
std,21.940734,0.344862,1.117357,15.759920
min,1.500300,3.000000,1.000000,6.963199
1%,2.019682,3.000000,2.000000,17.587666
5%,2.425582,4.000000,2.000000,22.914491
50%,4.140480,5.000000,3.000000,43.022177
95%,8.104892,5.000000,6.000000,75.225166
99%,9.783298,5.000000,7.000000,83.804339
max,15463.445700,5.000000,9.000000,86.656182


In [10]:
overlap

,flag_ipc_duplicate,flag_missing_items,flag_item_out_of_range,flag_speeding,flag_straightlining,flag_longstring,flag_mahalanobis,flag_screenw_outlier,flag_screenh_outlier,flag_testelapse_outlier
flag_ipc_duplicate,1.000000,0.004193,0.019142,0.025205,0.023482,0.029053,0.024945,-0.006425,-0.019504,0.027157
flag_missing_items,0.004193,1.000000,-0.016713,-0.004433,0.448900,-0.004342,-0.010623,-0.001318,-0.000158,-0.012993
flag_item_out_of_range,0.019142,-0.016713,1.000000,0.097949,0.048269,0.115483,-0.100924,-0.000288,0.001415,0.017557
flag_speeding,0.025205,-0.004433,0.097949,1.000000,0.413454,0.516326,0.040223,-0.000918,0.049719,0.008169
flag_straightlining,0.023482,0.448900,0.048269,0.413454,1.000000,0.492993,0.088157,-0.000563,0.000814,-0.010886
flag_longstring,0.029053,-0.004342,0.115483,0.516326,0.492993,1.000000,0.047268,-0.001720,-0.001735,-0.009003
flag_mahalanobis,0.024945,-0.010623,-0.100924,0.040223,0.088157,0.047268,1.000000,-0.001496,-0.002722,0.059010
flag_screenw_outlier,-0.006425,-0.001318,-0.000288,-0.000918,-0.000563,-0.001720,-0.001496,1.000000,0.166347,-0.003180
flag_screenh_outlier,-0.019504,-0.000158,0.001415,0.049719,0.000814,-0.001735,-0.002722,0.166347,1.000000,-0.005562
flag_testelapse_outlier,0.027157,-0.012993,0.017557,0.008169,-0.010886,-0.009003,0.059010,-0.003180,-0.005562,1.000000
